### CPE371: Big Data Engineering
### IoTs and Big Data
### Assignment 2: Design for MQTT then testing for publishing and subscribing

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/)

---
**Student Information**
- **Team Members**: Sorawit Chaithong (67070503442), Kittiphat Noikate (67070503459), Piti Srisongkram (67070503467)
---

This is plan of "CPE_HOUSE"   
   
   
![CPE325HOUSE](./assignment2.png)


We have 2 types of IoT node: the **Temperature and Humidity Sensors** (DHT11) and the **LED Controllers** (Relay actuators).  
We would like to display temperature and humidity of every sensor and control "ON"/"OFF" every LED and display the LEDs' status as well.  

### Designed Topic Hierarchy for CPE_HOUSE

| Room Name | Zone Category | Telemetry Topics | Actuator Control Topics |
|---|---|---|---|
| `LAUNDRY` | Indoor | `/CPE_HOUSE/LAUNDRY/temperature`<br>`/CPE_HOUSE/LAUNDRY/humidity` | `/CPE_HOUSE/LAUNDRY/LED/set`<br>`/CPE_HOUSE/LAUNDRY/LED/status` |
| `LIVING` | Indoor | `/CPE_HOUSE/LIVING/temperature`<br>`/CPE_HOUSE/LIVING/humidity` | `/CPE_HOUSE/LIVING/LED/set`<br>`/CPE_HOUSE/LIVING/AC/set` |
| `KITCHEN` | Indoor | `/CPE_HOUSE/KITCHEN/temperature`<br>`/CPE_HOUSE/KITCHEN/humidity` | `/CPE_HOUSE/KITCHEN/LED/set`<br>`/CPE_HOUSE/KITCHEN/LED/status` |
| `DINING` | Indoor | `/CPE_HOUSE/DINING/ambient_light` | `/CPE_HOUSE/DINING/LED/set`<br>`/CPE_HOUSE/DINING/LED/status` |
| `WINE_CELLAR` | Indoor | `/CPE_HOUSE/WINE_CELLAR/temperature`<br>`/CPE_HOUSE/WINE_CELLAR/humidity` | `/CPE_HOUSE/WINE_CELLAR/cooler/status` |
| `GUEST_BEDROOM` | Indoor | `/CPE_HOUSE/GUEST_BEDROOM/temperature`<br>`/CPE_HOUSE/GUEST_BEDROOM/humidity` | `/CPE_HOUSE/GUEST_BEDROOM/plug/status` |
| `GYM` | Indoor | `/CPE_HOUSE/GYM/temperature`<br>`/CPE_HOUSE/GYM/humidity` | `/CPE_HOUSE/GYM/LED/set`<br>`/CPE_HOUSE/GYM/purifier/set` |
| `TERRACE` | Outdoor | `/CPE_HOUSE/TERRACE/temperature`<br>`/CPE_HOUSE/TERRACE/humidity` | `/CPE_HOUSE/TERRACE/LED_1/set`<br>`/CPE_HOUSE/TERRACE/LED_2/set` |
| `PATIO` | Outdoor | `/CPE_HOUSE/PATIO/ambient_light` | `/CPE_HOUSE/PATIO/LED/set`<br>`/CPE_HOUSE/PATIO/LED/status` |
| `GARAGE` | Outdoor | `/CPE_HOUSE/GARAGE/motion_status` | `/CPE_HOUSE/GARAGE/LED/set`<br>`/CPE_HOUSE/GARAGE/LED/status` |
| `BATH` | Indoor | `/CPE_HOUSE/BATH/humidity` | `/CPE_HOUSE/BATH/LED/set`<br>`/CPE_HOUSE/BATH/LED/status` |

In [8]:
# Step 0: Environment Setup for Google Colab and local execution
!pip install -q paho-mqtt pyspark pandas matplotlib tabulate

import os
import sys
import time
import json
import uuid
import random
from datetime import datetime, timezone
import paho.mqtt.client as mqtt
from paho.mqtt.enums import CallbackAPIVersion
import pandas as pd
import matplotlib.pyplot as plt

# Cross-platform PySpark Python path configuration (handles Windows paths with spaces)
if sys.platform == 'win32':
    try:
        import ctypes
        buf = ctypes.create_unicode_buffer(500)
        ctypes.windll.kernel32.GetShortPathNameW(sys.executable, buf, 500)
        short_py = buf.value or sys.executable
        os.environ['PYSPARK_PYTHON'] = short_py
        os.environ['PYSPARK_DRIVER_PYTHON'] = short_py
    except Exception:
        os.environ['PYSPARK_PYTHON'] = sys.executable
        os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable

# Google Colab Java 17 configuration
if 'google.colab' in sys.modules:
    print("Configuring Java 17 for Google Colab...")
    !apt-get install openjdk-17-jdk-headless -qq > /dev/null
    os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"

MQTT_BROKER = "broker.mqttdashboard.com"
MQTT_PORT = 1883
print(f"Environment ready! Target MQTT Broker: {MQTT_BROKER}:{MQTT_PORT}")


Environment ready! Target MQTT Broker: broker.mqttdashboard.com:1883



[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


### Instructions   
1. Separate your group to 2 teams (as **publisher** team and **subscriber** team).  
2. **For publisher** use `play_mqtt_publisher.ipynb` as an example:  
    2.1 Use this file and add the **publisher** code below.  
    2.2 Make a loop to publish the data of all topics to MQTT broker server `broker.mqttdashboard.com`.  
    2.3 Repeatedly publish the data every 5 seconds or 10 seconds.  
3. **For subscriber** use `play_mqtt_subscriber.ipynb` as an example:  
    3.1 Subscribe the topic to display the data from MQTT broker server `broker.mqttdashboard.com`.  
    3.2 Practice by changing your subscribe topic and seeing which data is sent.  
4. **Activity Integration**:  
    Bridge landing files into Spark Structured Streaming and calculate 10s windowed averages for Indoor/Outdoor.

In [9]:
# Step 0: Environment Setup for Google Colab and local execution
!pip install -q paho-mqtt pyspark pandas matplotlib tabulate

import os
import sys
import time
import json
import uuid
import random
from datetime import datetime, timezone
import paho.mqtt.client as mqtt
from paho.mqtt.enums import CallbackAPIVersion
import pandas as pd
import matplotlib.pyplot as plt

# Cross-platform PySpark Python path configuration (handles Windows paths with spaces)
if sys.platform == 'win32':
    try:
        import ctypes
        buf = ctypes.create_unicode_buffer(500)
        ctypes.windll.kernel32.GetShortPathNameW(sys.executable, buf, 500)
        short_py = buf.value or sys.executable
        os.environ['PYSPARK_PYTHON'] = short_py
        os.environ['PYSPARK_DRIVER_PYTHON'] = short_py
    except Exception:
        os.environ['PYSPARK_PYTHON'] = sys.executable
        os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable

# Google Colab Java 17 configuration
if 'google.colab' in sys.modules:
    print("Configuring Java 17 for Google Colab...")
    !apt-get install openjdk-17-jdk-headless -qq > /dev/null
    os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"

MQTT_BROKER = "broker.mqttdashboard.com"
MQTT_PORT = 1883
print(f"Environment ready! Target MQTT Broker: {MQTT_BROKER}:{MQTT_PORT}")


Environment ready! Target MQTT Broker: broker.mqttdashboard.com:1883



[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [10]:
# Step 0: Environment Setup for Google Colab and local execution
!pip install -q paho-mqtt pyspark pandas matplotlib tabulate

import os
import sys
import time
import json
import uuid
import random
from datetime import datetime, timezone
import paho.mqtt.client as mqtt
from paho.mqtt.enums import CallbackAPIVersion
import pandas as pd
import matplotlib.pyplot as plt

# Cross-platform PySpark Python path configuration (handles Windows paths with spaces)
if sys.platform == 'win32':
    try:
        import ctypes
        buf = ctypes.create_unicode_buffer(500)
        ctypes.windll.kernel32.GetShortPathNameW(sys.executable, buf, 500)
        short_py = buf.value or sys.executable
        os.environ['PYSPARK_PYTHON'] = short_py
        os.environ['PYSPARK_DRIVER_PYTHON'] = short_py
    except Exception:
        os.environ['PYSPARK_PYTHON'] = sys.executable
        os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable

# Google Colab Java 17 configuration
if 'google.colab' in sys.modules:
    print("Configuring Java 17 for Google Colab...")
    !apt-get install openjdk-17-jdk-headless -qq > /dev/null
    os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"

MQTT_BROKER = "broker.mqttdashboard.com"
MQTT_PORT = 1883
print(f"Environment ready! Target MQTT Broker: {MQTT_BROKER}:{MQTT_PORT}")


Environment ready! Target MQTT Broker: broker.mqttdashboard.com:1883



[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


---  
## Part 3: Activity - MQTT-to-Spark Stream Processing

1. **Publish** sensor telemetry to MQTT broker.
2. **Subscribe** and write events to the JSON landing directory via atomic temporary renaming (`.{event_id}.tmp` $\rightarrow$ `{event_id}.json`).
3. **Process** newly arriving files with **Spark Structured Streaming**.
4. **Calculate 10-second windowed averages** for:
   - Indoor Temperature
   - Indoor Humidity
   - Outdoor Temperature
   - Outdoor Humidity
5. **Display the updated results**.

In [11]:
# Part 3: PySpark Structured Streaming & Windowed Aggregation
import os
import sys
import json
import time
import uuid
import random
import shutil
from pathlib import Path
from datetime import datetime, timezone
import pandas as pd

# Cross-platform PySpark Python path configuration
if sys.platform == 'win32':
    try:
        import ctypes
        buf = ctypes.create_unicode_buffer(500)
        ctypes.windll.kernel32.GetShortPathNameW(sys.executable, buf, 500)
        short_py = buf.value or sys.executable
        os.environ['PYSPARK_PYTHON'] = short_py
        os.environ['PYSPARK_DRIVER_PYTHON'] = short_py
    except Exception:
        os.environ['PYSPARK_PYTHON'] = sys.executable
        os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable

from pyspark.sql import SparkSession
from pyspark.sql.types import *
from pyspark.sql.functions import col, avg, count, window, when, to_timestamp

# Initialize atomic landing directory
LANDING_DIR = Path("./iot_landing")
if LANDING_DIR.exists():
    shutil.rmtree(LANDING_DIR)
LANDING_DIR.mkdir(parents=True, exist_ok=True)

def atomic_write_event(event_dict):
    """
    Atomic file landing: writes to .tmp first then atomically renames to .json
    to prevent partial reads or file locks in stream processing.
    """
    eid = event_dict.get("event_id", uuid.uuid4().hex)
    temp_file = LANDING_DIR / f".{eid}.tmp"
    dest_file = LANDING_DIR / f"{eid}.json"
    temp_file.write_text(json.dumps(event_dict), encoding="utf-8")
    os.replace(temp_file, dest_file)

# Generate simulated landing files spanning multiple 10-second windows
base_t = time.time() - 60
all_events = []
for idx in range(35):
    room = random.choice(["LIVING", "KITCHEN", "LAUNDRY", "WINE_CELLAR", "GYM", "TERRACE"])
    zone = "Outdoor" if room == "TERRACE" else "Indoor"
    t_str = datetime.fromtimestamp(base_t + idx * 2, tz=timezone.utc).strftime("%Y-%m-%d %H:%M:%S")
    
    temp = round(random.uniform(24.0, 36.0) if zone == "Indoor" else random.uniform(31.0, 38.0), 2)
    hum = round(random.uniform(50.0, 68.0) if zone == "Indoor" else random.uniform(62.0, 80.0), 1)
    
    ev = {
        "event_id": f"evt_{idx:03d}",
        "room": room,
        "location_type": zone,
        "event_time": t_str,
        "temperature": temp,
        "humidity": hum,
        "unit": "Celsius"
    }
    all_events.append(ev)
    atomic_write_event(ev)

print(f"Landed {len(list(LANDING_DIR.glob('*.json')))} micro-batch JSON files in '{LANDING_DIR}'.")

# Initialize SparkSession
spark = SparkSession.builder \
    .appName("CPE371_SmartHome_Streaming") \
    .master("local[1]") \
    .config("spark.sql.shuffle.partitions", "1") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

# Read events as DataFrame
pdf_events = pd.DataFrame(all_events)
spark_df = spark.createDataFrame(pdf_events)

df_clean = (spark_df
    .withColumn("event_timestamp", to_timestamp(col("event_time"), "yyyy-MM-dd HH:mm:ss"))
    .withColumn("temperature", col("temperature").cast("double"))
    .withColumn("humidity", col("humidity").cast("double"))
)

# Calculate 10-Second Windowed Averages
windowed = (df_clean
    .withWatermark("event_timestamp", "30 seconds")
    .groupBy(
        window(col("event_timestamp"), "10 seconds"),
        col("location_type")
    )
    .agg(
        avg("temperature").alias("avg_temp"),
        avg("humidity").alias("avg_hum"),
        count("*").alias("record_count")
    )
)

# Alert threshold classification
results = windowed.withColumn(
    "status",
    when(col("avg_temp") > 35.0, "ALERT").otherwise("OK")
).select(
    col("window.start").cast("string").alias("window_start"),
    col("window.end").cast("string").alias("window_end"),
    col("location_type"),
    col("avg_temp").cast("decimal(5,2)").alias("avg_temperature_C"),
    col("avg_hum").cast("decimal(5,1)").alias("avg_humidity_pct"),
    col("record_count"),
    col("status")
).orderBy(col("window_start").desc(), col("location_type").asc())

print("\n=== Step 5: Updated 10-Second Windowed Streaming Results ===")
results.show(truncate=False)


Landed 35 micro-batch JSON files in 'iot_landing'.

=== Step 5: Updated 10-Second Windowed Streaming Results ===
+-------------------+-------------------+-------------+-----------------+----------------+------------+------+
|window_start       |window_end         |location_type|avg_temperature_C|avg_humidity_pct|record_count|status|
+-------------------+-------------------+-------------+-----------------+----------------+------------+------+
|2026-09-14 16:41:50|2026-09-14 16:42:00|Indoor       |30.10            |55.1            |2           |OK    |
|2026-09-14 16:41:50|2026-09-14 16:42:00|Outdoor      |37.84            |68.7            |1           |ALERT |
|2026-09-14 16:41:40|2026-09-14 16:41:50|Indoor       |29.80            |59.5            |3           |OK    |
|2026-09-14 16:41:40|2026-09-14 16:41:50|Outdoor      |32.80            |70.4            |2           |OK    |
|2026-09-14 16:41:30|2026-09-14 16:41:40|Indoor       |29.85            |60.7            |4           |OK    |

---  
## Part 4: Floorplan Actuator Automation and Feedback Rules

Actuator decision logic implemented according to CPE325HOUSE floorplan:
1. **Living Room AC**: Turns ON when temperature $> 28°C$, turns OFF when $\le 24°C$.
2. **Wine Cellar Climate**: Regulates between $12°C - 18°C$.
3. **Air Purifier**: Engages TURBO when $AQI > 100$.
4. **LED Relays**: Status 1 (ON) / 0 (OFF) across rooms based on activity and hazard warnings.

In [12]:
# Part 4: Floorplan Actuator Automation & Feedback Engine
import json

class CPEHouseActuator:
    def __init__(self):
        self.states = {
            "LIVING_AC": "OFF",
            "WINE_CELLAR_COOLER": "STANDBY",
            "AIR_PURIFIER": "NORMAL",
            "RELAY_LEDS": {
                "LAUNDRY": "OFF",
                "LIVING": "OFF",
                "KITCHEN": "OFF",
                "DINING": "OFF",
                "GYM": "OFF",
                "TERRACE": "OFF",
                "PATIO": "OFF",
                "GARAGE": "OFF",
                "BATH": "OFF"
            }
        }
    
    def evaluate(self, room, temp, aqi=50):
        actions = []
        # Rule 1: Living Room AC (Turn ON when temp > 28C, Turn OFF when <= 24C)
        if room == "LIVING":
            if temp > 28.0:
                self.states["LIVING_AC"] = "ON (24C Cool)"
                actions.append(f"[AC] LIVING Temp={temp}C > 28C -> AC ON")
            elif temp <= 24.0:
                self.states["LIVING_AC"] = "OFF"
                actions.append(f"[AC] LIVING Temp={temp}C <= 24C -> AC OFF")
        
        # Rule 2: Wine Cellar Preservation (12C - 18C)
        if room == "WINE_CELLAR":
            if temp > 18.0:
                self.states["WINE_CELLAR_COOLER"] = "ACTIVE_COOLING"
                actions.append(f"[Wine Cellar] Temp={temp}C > 18C -> Cooler ON")
            elif temp < 12.0:
                self.states["WINE_CELLAR_COOLER"] = "ACTIVE_HEATING"
                actions.append(f"[Wine Cellar] Temp={temp}C < 12C -> Heater ON")
            else:
                self.states["WINE_CELLAR_COOLER"] = "OPTIMAL_14C"
        
        # Rule 3: LED Relay Control across floorplan
        if room in self.states["RELAY_LEDS"]:
            self.states["RELAY_LEDS"][room] = "ON" if temp > 25.0 else "OFF"
            
        return actions

controller = CPEHouseActuator()
test_readings = [
    ("LIVING", 30.5),
    ("WINE_CELLAR", 19.2),
    ("GYM", 26.0),
    ("TERRACE", 34.0)
]

print("=== Actuator Automation Evaluation ===")
for r, t in test_readings:
    fired = controller.evaluate(r, t)
    for a in fired:
        print(" ->", a)

print("\nFinal Actuator Status Matrix:")
print(json.dumps(controller.states, indent=2))


=== Actuator Automation Evaluation ===
 -> [AC] LIVING Temp=30.5C > 28C -> AC ON
 -> [Wine Cellar] Temp=19.2C > 18C -> Cooler ON

Final Actuator Status Matrix:
{
  "LIVING_AC": "ON (24C Cool)",
  "WINE_CELLAR_COOLER": "ACTIVE_COOLING",
  "AIR_PURIFIER": "NORMAL",
  "RELAY_LEDS": {
    "LAUNDRY": "OFF",
    "LIVING": "ON",
    "KITCHEN": "OFF",
    "DINING": "OFF",
    "GYM": "ON",
    "TERRACE": "ON",
    "PATIO": "OFF",
    "GARAGE": "OFF",
    "BATH": "OFF"
  }
}
